# Exploratory Data Analysis

## Key Findings

1.
2.
3.

## Imports

In [2]:
import os

import pandas as pd
import numpy as np

## Pre-Cleaning Checks

This section will inform which tasks the data-cleaning.py file needs to perform in order to get the data ready for the graph database.

### Findings

1. Raw file contains 55 reporters, each with a world aggregate row and partner rows (where available).
2. 11% of rows are world-total or other aggregate rows and will be excluded from edge construction to avoid double counting.
3. FOB value has over 70% of its data missing. The FOB value is mainly important to exports so this column can be dropped.
4. 32 reporters have a world total but no partners. They will be included in the graph as nodes but will not have any outgoing relationships.

In [7]:
raw = pd.read_csv("../data/raw/comtrade_gallium_imports_2025_raw.csv")

pd.set_option('display.max_columns', None)
raw.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,flowDesc,partnerCode,partnerISO,partnerDesc,partner2Code,partner2ISO,partner2Desc,classificationCode,classificationSearchCode,isOriginalClassification,cmdCode,cmdDesc,aggrLevel,isLeaf,customsCode,customsDesc,mosCode,motCode,motDesc,qtyUnitCode,qtyUnitAbbr,qty,isQtyEstimated,altQtyUnitCode,altQtyUnitAbbr,altQty,isAltQtyEstimated,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20250101,2025,52,2025,842,USA,USA,M,Import,0,W00,World,0,W00,World,H6,HS,True,811292,"Gallium, germanium, indium, niobium (columbium...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,1912812.0,False,8,kg,1912812.0,False,1912693.0,False,0.0,False,208883040.0,207646575.0,208883040.0,0,False,True
1,C,A,20250101,2025,52,2025,842,USA,USA,M,Import,40,AUT,Austria,0,W00,World,H6,HS,True,811292,"Gallium, germanium, indium, niobium (columbium...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,6470.0,False,8,kg,6470.0,False,6470.0,False,0.0,False,328547.0,325461.0,328547.0,0,True,False
2,C,A,20250101,2025,52,2025,842,USA,USA,M,Import,56,BEL,Belgium,0,W00,World,H6,HS,True,811292,"Gallium, germanium, indium, niobium (columbium...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,10935.0,False,8,kg,10935.0,False,10935.0,False,0.0,False,13982700.0,13910992.0,13982700.0,0,True,False
3,C,A,20250101,2025,52,2025,842,USA,USA,M,Import,76,BRA,Brazil,0,W00,World,H6,HS,True,811292,"Gallium, germanium, indium, niobium (columbium...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,1329811.0,False,8,kg,1329811.0,False,1329811.0,False,0.0,False,64630750.0,64269892.0,64630750.0,0,True,False
4,C,A,20250101,2025,52,2025,842,USA,USA,M,Import,124,CAN,Canada,0,W00,World,H6,HS,True,811292,"Gallium, germanium, indium, niobium (columbium...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,77848.0,False,8,kg,77848.0,False,77732.0,False,0.0,False,19836079.0,19799580.0,19836079.0,0,True,False


In [ ]:
# Check for unique values in columns of interest

print(f"Rows: {len(raw)}")
print(f"Reporters: {raw['reporterDesc'].nunique()}\n {sorted(raw['reporterDesc'].unique().tolist())}")
print(f"Partners: {raw['partnerDesc'].nunique()}\n {sorted(raw['partnerDesc'].unique().tolist())}")
print(f"Commodity Codes: {sorted(raw['cmdCode'].unique().tolist())}")
print(f"Flow types: {sorted(raw['flowDesc'].unique().tolist())}")
print(f"Reference Years: {sorted(raw['refYear'].unique().tolist())}")

Rows: 479
Reporters: 55
 ['Australia', 'Austria', 'Azerbaijan', 'Belgium', 'Brazil', 'Canada', 'Chile', 'China, Hong Kong SAR', 'Colombia', 'Croatia', 'Czechia', 'Denmark', 'Egypt', 'El Salvador', 'Estonia', 'European Union', 'Faroe Isds', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Latvia', 'Lithuania', 'Luxembourg', 'Malaysia', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Philippines', 'Poland', 'Portugal', 'Rep. of Korea', 'Rep. of Moldova', 'Romania', 'Serbia', 'Singapore', 'Slovakia', 'Slovenia', 'South Africa', 'Spain', 'Sri Lanka', 'Sweden', 'Switzerland', 'Türkiye', 'USA', 'United Kingdom']
Partners: 55
 ['Areas, nes', 'Armenia', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'China, Hong Kong SAR', 'Colombia', 'Congo', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France', 'Germany', 'Hungary', 'India', 'Ireland', 'Israel',

In [ ]:
# Check which rows are aggregates or self-trades and how many are left after filtering them out

world_mask = raw['partnerDesc'].str.strip().str.lower() == 'world'
agg_mask = raw['isAggregate'] == True
self_mask = raw['reporterDesc'] == raw['partnerDesc']

print(f"World total rows: {world_mask.sum()}, ({world_mask.sum() / len(raw) * 100:.2f}% of raw rows)")
print(f"Other aggregate rows (non-world): {((agg_mask) & (~world_mask)).sum()}")
print(f"Self-trade rows: {self_mask.sum()}")
print(f"Rows remaining after filtering world, aggregate, and self-trade rows: {((~world_mask) & (~agg_mask) & (~self_mask)).sum()}")


World total rows: 55, (11.48% of raw rows)
Other aggregate rows (non-world): 253
Self-trade rows: 2
Rows remaining after filtering world, aggregate, and self-trade rows: 171


In [11]:
# Count missing values

null_counts = raw.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

print(f"Columns with missing values:\n {null_counts}")
print(f"Percentage of missing values per column:\n {null_counts / len(raw) * 100}")

Columns with missing values:
 fobvalue          355
altQtyUnitAbbr     95
cifvalue           21
dtype: int64
Percentage of missing values per column:
 fobvalue          74.112735
altQtyUnitAbbr    19.832985
cifvalue           4.384134
dtype: float64


In [18]:
# Check which reporters have no partner data

no_partner_data = raw[raw['partnerDesc'].isnull()]['reporterDesc'].unique()
no_partner_data

<StringArray>
[]
Length: 0, dtype: str

In [23]:
world_totals = (
    raw[world_mask]
    .groupby("reporterDesc", as_index=False)["primaryValue"]
    .sum()
    .rename(columns={"primaryValue": "world_total_value"})
)

partner_sums = (
    raw[~world_mask & ~agg_mask & ~self_mask]
    .groupby("reporterDesc", as_index=False)["primaryValue"]
    .sum()
    .rename(columns={"primaryValue": "sum_of_partners_value"})
)

mismatch = world_totals.merge(partner_sums, on="reporterDesc", how="left")

print(f"Reporters with world total but no partner data: {mismatch[mismatch['sum_of_partners_value'].isna()]['reporterDesc'].tolist()}")
print(f"There are {len(mismatch[mismatch['sum_of_partners_value'].isna()])} reporters with world total but no partner data.")

Reporters with world total but no partner data: ['Australia', 'Azerbaijan', 'Brazil', 'Canada', 'Colombia', 'Czechia', 'Estonia', 'European Union', 'Faroe Isds', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'India', 'Indonesia', 'Latvia', 'Luxembourg', 'Malaysia', 'Norway', 'Portugal', 'Rep. of Moldova', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'South Africa', 'Spain', 'Sweden', 'Switzerland', 'Türkiye', 'United Kingdom']
There are 32 reporters with world total but no partner data.


## Exploratory Data Analysis